In [1]:
import pandas as pd
import numpy as np

file_path = "../data/processed/online_retail_clean.csv"

df = pd.read_csv(file_path)

df.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,transaction_type,revenue,sales_revenue
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,SALE,15.30,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,SALE,20.34,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,SALE,22.00,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,SALE,20.34,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,SALE,20.34,20.34


In [5]:
customer_sales = (
    df[
        (df["transaction_type"] == "SALE") &
        (df["customer_id"].notna())
    ]
    .groupby("customer_id", as_index=False)
    .agg(
        total_revenue=("sales_revenue", "sum"),
        total_orders=("invoice_no", "nunique"),
        total_units=("quantity", "sum")
    )
)

customer_sales.head()

,customer_id,total_revenue,total_orders,total_units
0,12346.0,77183.60,1,74215
1,12347.0,4310.00,7,2458
2,12348.0,1797.24,4,2341
3,12349.0,1757.55,1,631
4,12350.0,334.40,1,197


In [6]:
customer_sales.shape

(4339, 4)

In [7]:

customer_sales["total_revenue"].describe()

count      4339.000000
mean       2048.215924
std        8984.248352
min           0.000000
25%         306.455000
50%         668.560000
75%        1660.315000
max      280206.020000
Name: total_revenue, dtype: float64

In [8]:
customer_sales[customer_sales["total_revenue"] == 0].shape

(1, 4)

In [9]:
customer_sales[
    customer_sales["total_revenue"] == 0
].head()

,customer_id,total_revenue,total_orders,total_units
685,13256.0,0.0,1,12540


In [10]:
df[df["customer_id"] == 685][
    [
        "invoice_no",
        "stock_code",
        "quantity",
        "unit_price",
        "transaction_type",
        "sales_revenue"
    ]
].head(20)

,invoice_no,stock_code,quantity,unit_price,transaction_type,sales_revenue


In [11]:
df[df["customer_id"] == 685]["transaction_type"].value_counts()

Series([], Name: count, dtype: int64)

In [12]:
customer_sales[customer_sales["customer_id"] == 685]

,customer_id,total_revenue,total_orders,total_units


In [13]:
customer_sales.dtypes

customer_id      float64
total_revenue    float64
total_orders       int64
total_units        int64
dtype: object

In [14]:
customer_sales[
    customer_sales["total_revenue"] <= 0
]

,customer_id,total_revenue,total_orders,total_units
685,13256.0,0.0,1,12540


In [15]:
df[df["customer_id"] == 13256][
    [
        "invoice_no",
        "stock_code",
        "description",
        "quantity",
        "unit_price",
        "transaction_type",
        "revenue",
        "sales_revenue"
    ]
]

,invoice_no,stock_code,description,quantity,unit_price,transaction_type,revenue,sales_revenue
497394,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,0.0,SALE,0.0,0.0


In [16]:
df[df["customer_id"] == 13256]["transaction_type"].value_counts()

transaction_type
SALE    1
Name: count, dtype: int64

In [6]:
sales_df = df[
    (df["transaction_type"] == "SALE") &
    (df["unit_price"] > 0) &
    (df["customer_id"].notna())
].copy()

sales_df.shape

(392692, 11)

In [18]:
customer_sales = (
    sales_df
    .groupby("customer_id", as_index=False)
    .agg(
        total_revenue=("sales_revenue", "sum"),
        total_orders=("invoice_no", "nunique"),
        total_units=("quantity", "sum")
    )
)

customer_sales.head()

,customer_id,total_revenue,total_orders,total_units
0,12346.0,77183.60,1,74215
1,12347.0,4310.00,7,2458
2,12348.0,1797.24,4,2341
3,12349.0,1757.55,1,631
4,12350.0,334.40,1,197


In [19]:
customer_sales["total_revenue"].describe()

count      4338.000000
mean       2048.688081
std        8985.230220
min           3.750000
25%         306.482500
50%         668.570000
75%        1660.597500
max      280206.020000
Name: total_revenue, dtype: float64

In [20]:
customer_sales_sorted = customer_sales.sort_values(
    "total_revenue",
    ascending=False
).reset_index(drop=True)

total_customer_revenue = customer_sales_sorted["total_revenue"].sum()

customer_sales_sorted["cumulative_revenue"] = (
    customer_sales_sorted["total_revenue"].cumsum()
)

customer_sales_sorted["cumulative_revenue_share"] = (
    customer_sales_sorted["cumulative_revenue"]
    / total_customer_revenue
)

n_customers = len(customer_sales_sorted)

for pct in [0.01, 0.05, 0.10]:
    n = int(n_customers * pct)
    revenue_share = (
        customer_sales_sorted.iloc[:n]["total_revenue"].sum()
        / total_customer_revenue
        * 100
    )
    
    print(f"Top {pct*100:.0f}% ({n} customers): {revenue_share:.2f}% of revenue")

Top 1% (43 customers): 31.84% of revenue
Top 5% (216 customers): 50.39% of revenue
Top 10% (433 customers): 61.41% of revenue


In [21]:
customer_sales["total_orders"].describe()

count    4338.000000
mean        4.272015
std         7.697998
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       209.000000
Name: total_orders, dtype: float64

In [22]:
customer_sales["total_orders"].value_counts().sort_index().head(20)

total_orders
1     1493
2      835
3      508
4      388
5      242
6      172
7      143
8       98
9       68
10      54
11      52
12      45
13      30
14      20
15      28
16      11
17      18
18      14
19      12
20      12
Name: count, dtype: int64

In [23]:
print("One-order customers:",
      (customer_sales["total_orders"] == 1).sum())

print("Repeat customers:",
      (customer_sales["total_orders"] > 1).sum())

One-order customers: 1493
Repeat customers: 2845


In [24]:
customer_sales["avg_order_value"] = (
    customer_sales["total_revenue"]
    / customer_sales["total_orders"]
)

customer_sales["avg_order_value"].describe()


count     4338.000000
mean       417.645735
std       1796.511343
min          3.450000
25%        177.867083
50%        291.940000
75%        428.280625
max      84236.250000
Name: avg_order_value, dtype: float64

In [25]:
customer_sales.sort_values(
    "avg_order_value",
    ascending=False
)[
    [
        "customer_id",
        "total_revenue",
        "total_orders",
        "total_units",
        "avg_order_value"
    ]
].head(10)

,customer_id,total_revenue,total_orders,total_units,avg_order_value
3008,16446.0,168472.50,2,80997,84236.250000
0,12346.0,77183.60,1,74215,77183.600000
2502,15749.0,44534.30,3,18028,14844.766667
2011,15098.0,39916.50,3,121,13305.500000
10,12357.0,6207.67,1,2708,6207.670000
55,12415.0,124914.53,21,77374,5948.310952
196,12590.0,9864.26,2,4283,4932.130000
278,12688.0,4873.81,1,3028,4873.810000
329,12752.0,4366.78,1,2262,4366.780000
4201,18102.0,259657.30,60,64124,4327.621667


In [26]:
customer_sales.groupby(
    customer_sales["total_orders"].eq(1).map({
        True: "One-time",
        False: "Repeat"
    })
)["avg_order_value"].describe()

,count,mean,std,min,25%,50%,75%,max
total_orders,,,,,,,,
One-time,1493.0,411.245520,2034.246264,3.75,152.650,255.91,406.090000,77183.60
Repeat,2845.0,421.004442,1658.549739,3.45,191.875,302.56,440.316667,84236.25


In [9]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"])

analysis_date = df["invoice_date"].max() + pd.Timedelta(days=1)

analysis_date

Timestamp('2011-12-10 12:50:00')

In [7]:
df["invoice_date"] = pd.to_datetime(df["invoice_date"])
sales_df["invoice_date"] = pd.to_datetime(sales_df["invoice_date"])

In [10]:
rfm = (
    sales_df
    .groupby("customer_id")
    .agg(
        recency=("invoice_date", lambda x: 
                 (analysis_date - x.max()).days),
        frequency=("invoice_no", "nunique"),
        monetary=("sales_revenue", "sum")
    )
    .reset_index()
)

rfm.head()

,customer_id,recency,frequency,monetary
0,12346.0,326,1,77183.60
1,12347.0,2,7,4310.00
2,12348.0,75,4,1797.24
3,12349.0,19,1,1757.55
4,12350.0,310,1,334.40


In [37]:
rfm.shape

(4338, 4)

In [38]:
rfm.describe()

,customer_id,recency,frequency,monetary
count,4338.000000,4338.000000,4338.000000,4338.000000
mean,15300.408022,92.536422,4.272015,2048.688081
std,1721.808492,100.014169,7.697998,8985.230220
min,12346.000000,1.000000,1.000000,3.750000
25%,13813.250000,18.000000,1.000000,306.482500
50%,15299.500000,51.000000,2.000000,668.570000
75%,16778.750000,142.000000,5.000000,1660.597500
max,18287.000000,374.000000,209.000000,280206.020000


In [11]:
rfm["R_score"] = pd.qcut(
    rfm["recency"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    5,
    labels=[1, 2, 3, 4, 5]
)

In [12]:
rfm["RFM_score"] = (
    rfm["R_score"].astype(int)
    + rfm["F_score"].astype(int)
    + rfm["M_score"].astype(int)
)

rfm.head()

,customer_id,recency,frequency,monetary,R_score,F_score,M_score,RFM_score
0,12346.0,326,1,77183.60,1,1,5,7
1,12347.0,2,7,4310.00,5,5,5,15
2,12348.0,75,4,1797.24,2,4,4,10
3,12349.0,19,1,1757.55,4,1,4,9
4,12350.0,310,1,334.40,1,1,2,4


In [13]:
rfm[["R_score", "F_score", "M_score", "RFM_score"]].describe()

,RFM_score
count,4338.000000
mean,9.015445
std,3.590394
min,3.000000
25%,6.000000
50%,9.000000
75%,12.000000
max,15.000000


In [14]:
print("Recency score distribution:")
print(rfm["R_score"].value_counts().sort_index())

print("\nFrequency score distribution:")
print(rfm["F_score"].value_counts().sort_index())

print("\nMonetary score distribution:")
print(rfm["M_score"].value_counts().sort_index())

Recency score distribution:
R_score
5    868
4    904
3    858
2    843
1    865
Name: count, dtype: int64

Frequency score distribution:
F_score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64

Monetary score distribution:
M_score
1    868
2    867
3    868
4    867
5    868
Name: count, dtype: int64


In [15]:
rfm.groupby("RFM_score").agg(
    customers=("customer_id", "count"),
    total_revenue=("monetary", "sum"),
    avg_revenue=("monetary", "mean")
).sort_index()

,customers,total_revenue,avg_revenue
RFM_score,,,
3,183,26408.940,144.311148
4,361,77720.610,215.292548
5,337,97494.910,289.302404
6,426,163001.321,382.632209
7,377,266962.930,708.124483
8,375,251787.412,671.433099
9,336,359285.370,1069.301696
10,342,356695.410,1042.969035
11,347,472046.771,1360.365334


In [16]:
rfm_summary = (
    rfm.groupby(["R_score", "F_score", "M_score"])
    .agg(
        customers=("customer_id", "count"),
        revenue=("monetary", "sum")
    )
    .reset_index()
)

rfm_summary.sort_values(
    ["M_score", "F_score", "R_score"],
    ascending=False
).head(20)

,R_score,F_score,M_score,customers,revenue
117,1,5,5,5,22106.630
93,2,5,5,28,117193.240
69,3,5,5,80,422757.770
46,4,5,5,177,1083424.740
23,5,5,5,347,3896691.140
113,1,4,5,3,7907.460
89,2,4,5,29,79607.310
66,3,4,5,40,127765.390
42,4,4,5,44,165558.770
18,5,4,5,41,135033.620


In [17]:
def classify_customer(row):
    if row["R_score"] >= 4 and row["F_score"] >= 4 and row["M_score"] >= 4:
        return "High-Value Active"

    elif row["F_score"] >= 4 and row["R_score"] >= 3:
        return "Loyal Repeat"

    elif row["R_score"] >= 4 and row["F_score"] <= 2:
        return "Recent / New"

    elif row["R_score"] <= 2 and row["M_score"] >= 4:
        return "High-Value At-Risk"

    elif row["R_score"] <= 2:
        return "Inactive"

    else:
        return "Developing"


rfm["segment"] = rfm.apply(classify_customer, axis=1)

In [18]:
segment_summary = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        total_revenue=("monetary", "sum"),
        avg_revenue=("monetary", "mean"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean")
    )
    .sort_values("total_revenue", ascending=False)
)

segment_summary

,customers,total_revenue,avg_revenue,avg_recency,avg_frequency
segment,,,,,
High-Value Active,957,5791640.740,6051.871202,12.824451,11.123302
Loyal Repeat,503,895651.561,1780.619406,38.023857,5.077535
High-Value At-Risk,337,828665.101,2458.946887,143.845697,3.753709
Developing,851,738322.190,867.593643,39.690952,1.850764
Inactive,1371,487709.562,355.732722,205.590080,1.529540
Recent / New,319,145219.740,455.234295,18.517241,1.241379


In [47]:
segment_rfm = (
    rfm.groupby("segment")
    .agg(
        customers=("customer_id", "count"),
        min_R=("R_score", "min"),
        max_R=("R_score", "max"),
        min_F=("F_score", "min"),
        max_F=("F_score", "max"),
        min_M=("M_score", "min"),
        max_M=("M_score", "max")
    )
)

segment_rfm

,customers,min_R,max_R,min_F,max_F,min_M,max_M
segment,,,,,,,
Developing,851,5,3,1,3,1,5
High-Value Active,957,5,4,4,5,4,5
High-Value At-Risk,337,2,1,1,5,4,5
Inactive,1371,2,1,1,5,1,3
Loyal Repeat,503,5,3,4,5,1,5
Recent / New,319,5,4,1,2,1,5


In [48]:
pd.crosstab(
    rfm["segment"],
    rfm["RFM_score"]
)

RFM_score,3,4,5,6,7,8,9,10,11,12,13,14,15
segment,,,,,,,,,,,,,
Developing,0,0,67,126,104,111,162,137,97,41,6,0,0
High-Value Active,0,0,0,0,0,0,0,0,0,118,192,300,347
High-Value At-Risk,0,0,0,17,28,50,75,88,51,28,0,0,0
Inactive,183,361,270,228,157,118,49,5,0,0,0,0,0
Loyal Repeat,0,0,0,0,0,5,11,82,184,133,88,0,0
Recent / New,0,0,0,55,88,91,39,30,15,1,0,0,0


In [49]:
correlation = customer_sales[
    ["total_orders", "total_revenue"]
].corr()

correlation

,total_orders,total_revenue
total_orders,1.00000,0.55278
total_revenue,0.55278,1.00000


In [50]:
customer_sales[
    ["total_orders", "total_revenue"]
].describe()

,total_orders,total_revenue
count,4338.000000,4338.000000
mean,4.272015,2048.688081
std,7.697998,8985.230220
min,1.000000,3.750000
25%,1.000000,306.482500
50%,2.000000,668.570000
75%,5.000000,1660.597500
max,209.000000,280206.020000


In [51]:
customer_sorted = customer_sales.sort_values(
    "total_revenue",
    ascending=False
).reset_index(drop=True)

total_customer_revenue = customer_sorted["total_revenue"].sum()

for pct in [0.01, 0.05, 0.10, 0.20]:
    n = int(len(customer_sorted) * pct)

    revenue_share = (
        customer_sorted.iloc[:n]["total_revenue"].sum()
        / total_customer_revenue
        * 100
    )

    print(
        f"Top {int(pct*100)}% customers: "
        f"{revenue_share:.2f}% of revenue"
    )

Top 1% customers: 31.84% of revenue
Top 5% customers: 50.39% of revenue
Top 10% customers: 61.41% of revenue
Top 20% customers: 74.66% of revenue


In [19]:
rfm_export = rfm[
    [
        "customer_id",
        "recency",
        "frequency",
        "monetary",
        "R_score",
        "F_score",
        "M_score",
        "RFM_score",
        "segment"
    ]
].copy()

rfm_export.to_csv(
    "../data/processed/customer_rfm.csv",
    index=False
)